In [1]:
import requests
import gzip
import io

In [2]:
# bring in https://www.minorplanetcenter.net/iau/ECS/MPCAT-OBS/midmonth/UnnObs.txt.gz and split on new lines
url = "https://www.minorplanetcenter.net/iau/ECS/MPCAT-OBS/midmonth/UnnObs.txt.gz"

# 1. Fetch the compressed data
response = requests.get(url)

if response.status_code == 200:
    # 2. Use io.BytesIO to treat the raw bytes as a file-like object
    # 3. Use gzip.GzipFile to decompress
    with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
        # Read the decompressed content and decode to string
        content = f.read().decode('utf-8')
    
    # 4. Split on new lines
    lines = content.splitlines()

    print(f"Successfully processed {len(lines)} lines.")
    # Show the first line as an example
    if lines:
        print("First line:", lines[0])
else:
    print(f"Failed to download file. Status code: {response.status_code}")

Successfully processed 25917404 lines.
First line:      I73O00A* A1873 07 30.31661 23 14 41.96 -01 41 52.7          12   V AN082767


In [3]:
# rename according to IDs
import requests
r = requests.get("https://www.minorplanetcenter.net/iau/ECS/MPCAT/current/ids.txt")
lookup = {}
for line in r.content.decode().splitlines():
    primary = line[0:7]
    for i in range(0,len(line),7):
        lookup[line[i:i+7]] = primary


r = requests.get("https://www.minorplanetcenter.net/iau/ECS/MPCAT/numids.txt")
for line in r.content.decode().splitlines():
    line = line[6:]
    primary = line[0:7]
    for i in range(0,len(line),7):
        lookup[line[i:i+7]] = primary

# now we can rename the designations in lines
renamed_lines = []
for line in lines:
    desig = line[5:12]
    if desig in lookup:
        new_desig = lookup[desig]
        renamed_line = line[:5] + new_desig + line[12:]
        renamed_lines.append(renamed_line)
    else:
        renamed_lines.append(line)
lines = renamed_lines

In [4]:
# import the astrometry_counter.py file
from astrometry_counter import process_astrometry

# refresh the astrometry_counter package and re-import the process_astrometry function
import importlib
import astrometry_counter
importlib.reload(astrometry_counter)
from astrometry_counter import process_astrometry

out = process_astrometry(lines)

In [6]:
# dump out to a json
import json
with open('astrometry_counts.json', 'w') as f:
    json.dump(out, f, indent=4)